# Fine-tuning de un modelo de lenguaje en español sobre los títulos

Este notebook entrena un modelo de lenguaje en español (por defecto **BETO**, `dccuchile/bert-base-spanish-wwm-uncased`) para predecir si una publicación es *used* a partir de su título. Está pensado para correr en Kaggle con GPU y no necesita los archivos `.py` del proyecto.

**Qué hace:**
1. Busca en `/kaggle/input` los datos de la competencia (`train_data.jsonlines`, `test_data.jsonlines`) y el archivo de folds del proyecto (`folds.csv`).
2. Para cada uno de los 5 folds de `folds.csv` (los mismos de la validación del proyecto):
   - ajusta el modelo con las otras 4 durante 3 épocas;
   - al final de cada época evalúa en el fold de validación y se queda con la época de menor log loss, como el early stopping del LightGBM;
   - con esa época predice el fold de validación (out-of-fold) y test.
3. Guarda las predicciones de cada fold apenas termina, en `/kaggle/working/`. Al final arma:
   - `finetune_titulo_oof.csv` (fila, fold, proba_used): probabilidad de used out-of-fold de cada fila de train;
   - `finetune_titulo_test.csv` (fila, proba_used): promedio de los 5 modelos de fold para cada fila de test.
4. Compara con el submodelo TF-IDF del proyecto: en total y en vendedores nuevos con una sola publicación.

**Cuánto tarda:** con una GPU T4, ~6-10 minutos por fold (3 épocas sobre 56.000 títulos, con evaluación en validación y test en cada época), **~35-50 minutos en total**, más ~1-2 minutos de descarga del modelo. Con `TEXTO = "titulo_warranty"` los textos son más largos, pero el `max_length` es el mismo (48 tokens), así que tarda parecido.

**Parámetros:** se cambian en la celda siguiente (modelo, texto de entrada, tamaño del lote, etc.).

In [ ]:
# ---- Parámetros ----
MODELO = "dccuchile/bert-base-spanish-wwm-uncased"
# Alternativas si BETO falla al descargar: "bertin-project/bertin-roberta-base-spanish" (RoBERTa en español)
# o "dccuchile/bert-base-spanish-wwm-cased". "PlanTL-GOB-ES/roberta-base-bne" ya no sirve: el repositorio
# fue discontinuado y no tiene los pesos.
TEXTO = "titulo"  # "titulo" o "titulo_warranty" (título + " [SEP] " + warranty)
MAX_LENGTH = 48
BATCH = 64  # si se queda sin memoria de GPU, bajarlo a 32
LEARNING_RATE = 3e-5
EPOCAS = 3
WARMUP = 0.10
WEIGHT_DECAY = 0.01
FP16 = True
SEMILLA = 42
RUTA_ENTRADA = "/kaggle/input"
RUTA_SALIDA = "/kaggle/working"
REQUERIR_GPU = True
MAX_FILAS = None  # solo para pruebas: limita las filas de train y test

# Referencia: submodelo TF-IDF del proyecto (proba_used_titulo, out-of-fold con los mismos folds)
REFERENCIA_TFIDF = {
    "total": {"accuracy": 0.822, "auc": 0.898},
    "nuevo con una publicación (train + test)": {"accuracy": 0.750, "auc": 0.836},
    "nuevo con una publicación en su fold (definición del proyecto)": {"accuracy": 0.756, "auc": 0.840},
}

In [ ]:
import glob
import json
import os
import random
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

assert TEXTO in ("titulo", "titulo_warranty"), 'TEXTO tiene que ser "titulo" o "titulo_warranty"'
hay_gpu = torch.cuda.is_available()
print(f"torch {torch.__version__} | GPU disponible: {hay_gpu}"
      + (f" ({torch.cuda.get_device_name(0)})" if hay_gpu else ""))
if REQUERIR_GPU and not hay_gpu:
    raise RuntimeError("No hay GPU. En el panel derecho del notebook: Settings > Accelerator > GPU T4 x2 (o GPU T4), "
                       "y volvé a ejecutar. Sin GPU el entrenamiento tardaría muchas horas.")
DISPOSITIVO = torch.device("cuda" if hay_gpu else "cpu")
os.makedirs(RUTA_SALIDA, exist_ok=True)
USAR_FP16 = FP16 and hay_gpu

## Datos

Se buscan los archivos con `glob` en cualquier subcarpeta de `/kaggle/input`, porque la ruta exacta depende de cómo se agregaron la competencia y el dataset.

In [ ]:
def buscar(nombre):
    encontrados = sorted(glob.glob(os.path.join(RUTA_ENTRADA, "**", nombre), recursive=True))
    if not encontrados:
        contenido = sorted(glob.glob(os.path.join(RUTA_ENTRADA, "**", "*"), recursive=True))[:100]
        raise FileNotFoundError(
            f"No se encontró {nombre} dentro de {RUTA_ENTRADA}. Revisá que la competencia y el dataset con folds.csv "
            f"estén agregados como datos de entrada (panel derecho: Input > Add Input). Lo que hay ahora:\n"
            + ("\n".join(contenido) if contenido else "(vacío)"))
    if len(encontrados) > 1:
        print(f"Aviso: hay {len(encontrados)} archivos {nombre}; se usa {encontrados[0]}")
    return encontrados[0]


def leer_jsonlines(ruta, columnas):
    filas = []
    with open(ruta, encoding="utf-8") as f:
        for linea in f:
            if linea.strip():
                registro = json.loads(linea)
                filas.append({c: registro.get(c) for c in columnas})
                if MAX_FILAS and len(filas) >= MAX_FILAS:
                    break
    return pd.DataFrame(filas)


RUTA_TRAIN, RUTA_TEST, RUTA_FOLDS = buscar("train_data.jsonlines"), buscar("test_data.jsonlines"), buscar("folds.csv")
print(f"train: {RUTA_TRAIN}\ntest: {RUTA_TEST}\nfolds: {RUTA_FOLDS}")
train = leer_jsonlines(RUTA_TRAIN, ["title", "warranty", "seller_id", "condition"])
test = leer_jsonlines(RUTA_TEST, ["title", "warranty", "seller_id"])
folds = pd.read_csv(RUTA_FOLDS)
if MAX_FILAS:
    folds = folds[folds["fila"] < len(train)]
assert list(folds.columns) == ["fila", "fold"], "folds.csv tiene que tener las columnas fila y fold"
assert len(folds) == len(train) and (folds["fila"].to_numpy() == np.arange(len(train))).all(), \
    "folds.csv no corresponde a train_data.jsonlines (distinta cantidad u orden de filas)"
y = (train["condition"] == "used").astype(int).to_numpy()
FOLD = folds["fold"].to_numpy()
print(f"train {len(train):,} filas ({y.mean():.1%} used) | test {len(test):,} filas | "
      f"filas por fold: {np.bincount(FOLD)[1:].tolist()}")

### Texto de entrada

El título original, con los caracteres con problemas de codificación corregidos ("ã¿" → "ó", "ã\x8d" → "í"). Con `TEXTO = "titulo_warranty"` se agrega `" [SEP] "` y el texto de warranty (vacío si no hay).

BETO *uncased* se entrenó en minúsculas (conserva las tildes), así que si el nombre del modelo dice `uncased` el texto se pasa a minúsculas antes de tokenizar; el tokenizador de BETO igual lo hace solo. Se muestra cuántos tokens desconocidos quedan y qué porcentaje de textos se corta en los 48 tokens.

In [ ]:
REPARACIONES = {"ã¿": "ó", "ã\x8d": "í"}


def limpiar(texto):
    texto = texto if isinstance(texto, str) else ""
    for malo, bueno in REPARACIONES.items():
        texto = texto.replace(malo, bueno)
    return texto.strip()


def armar_texto(df):
    textos = df["title"].map(limpiar)
    if TEXTO == "titulo_warranty":
        textos = textos + " [SEP] " + df["warranty"].map(limpiar)
    if "uncased" in MODELO.lower():
        textos = textos.str.lower()
    return textos.tolist()


textos_train, textos_test = armar_texto(train), armar_texto(test)
tokenizador = AutoTokenizer.from_pretrained(MODELO)


def tokenizar(textos):
    codificado = tokenizador(textos, truncation=True, max_length=MAX_LENGTH, padding="max_length", return_tensors="pt")
    return codificado["input_ids"], codificado["attention_mask"]


ids_train, mascara_train = tokenizar(textos_train)
ids_test, mascara_test = tokenizar(textos_test)
largo = mascara_train.sum(1)
desconocidos = (ids_train == tokenizador.unk_token_id).sum().item() if tokenizador.unk_token_id is not None else 0
print(f"Ejemplo: {textos_train[0]!r} -> {tokenizador.convert_ids_to_tokens(ids_train[0][:largo[0]])}")
print(f"Tokens por texto: mediana {largo.median().item():.0f}, máximo {largo.max().item()} | "
      f"textos cortados en {MAX_LENGTH}: {(largo == MAX_LENGTH).float().mean().item():.1%} | "
      f"tokens desconocidos: {desconocidos / largo.sum().item():.2%}")

## Entrenamiento por fold

Para cada fold: AdamW (`weight_decay` 0,01, sin aplicarlo a sesgos ni LayerNorm), learning rate 3e-5 con *warmup* lineal del 10% y bajada lineal, lotes de 64, 3 épocas, fp16 y semilla 42.

Al final de cada época se predice el fold de validación y test. Se guardan las predicciones de la época con menor log loss en validación.

Cada fold guarda `fold_{k}_oof.csv` y `fold_{k}_test.csv` en `/kaggle/working` apenas termina. Si se vuelve a ejecutar en la **misma sesión**, los folds ya guardados se leen y no se reentrenan. Una sesión nueva de Kaggle empieza con `/kaggle/working` vacío.

In [ ]:
def fijar_semilla(semilla):
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    torch.cuda.manual_seed_all(semilla)


def predecir(modelo, ids, mascara, lote=256):
    modelo.eval()
    probas = []
    with torch.no_grad():
        for i in range(0, len(ids), lote):
            with torch.autocast(device_type=DISPOSITIVO.type, dtype=torch.float16, enabled=USAR_FP16):
                logits = modelo(input_ids=ids[i:i + lote].to(DISPOSITIVO),
                                attention_mask=mascara[i:i + lote].to(DISPOSITIVO)).logits
            probas.append(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(probas)


def entrenar_fold(k):
    fijar_semilla(SEMILLA)
    idx_tr, idx_val = np.flatnonzero(FOLD != k), np.flatnonzero(FOLD == k)
    modelo = AutoModelForSequenceClassification.from_pretrained(MODELO, num_labels=2).to(DISPOSITIVO)
    sin_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
    grupos = [
        {"params": [p for n, p in modelo.named_parameters() if not any(s in n for s in sin_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in modelo.named_parameters() if any(s in n for s in sin_decay)], "weight_decay": 0.0},
    ]
    optimizador = torch.optim.AdamW(grupos, lr=LEARNING_RATE)
    pasos_por_epoca = int(np.ceil(len(idx_tr) / BATCH))
    programa = get_linear_schedule_with_warmup(optimizador, int(WARMUP * pasos_por_epoca * EPOCAS), pasos_por_epoca * EPOCAS)
    escalador = torch.amp.GradScaler("cuda", enabled=USAR_FP16)
    generador = torch.Generator().manual_seed(SEMILLA)
    etiquetas = torch.tensor(y, dtype=torch.long)

    mejor = {"log_loss": np.inf}
    historia = []
    for epoca in range(1, EPOCAS + 1):
        modelo.train()
        inicio = time.time()
        orden = idx_tr[torch.randperm(len(idx_tr), generator=generador).numpy()]
        perdida_total = 0.0
        for paso in range(pasos_por_epoca):
            lote = orden[paso * BATCH:(paso + 1) * BATCH]
            with torch.autocast(device_type=DISPOSITIVO.type, dtype=torch.float16, enabled=USAR_FP16):
                salida = modelo(input_ids=ids_train[lote].to(DISPOSITIVO), attention_mask=mascara_train[lote].to(DISPOSITIVO),
                                labels=etiquetas[lote].to(DISPOSITIVO))
            optimizador.zero_grad(set_to_none=True)
            escalador.scale(salida.loss).backward()
            escalador.unscale_(optimizador)
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            escalador.step(optimizador)
            escalador.update()
            programa.step()
            perdida_total += salida.loss.item()
        proba_val = predecir(modelo, ids_train[idx_val], mascara_train[idx_val])
        perdida_val = log_loss(y[idx_val], np.clip(proba_val, 1e-7, 1 - 1e-7), labels=[0, 1])
        historia.append({"época": epoca, "log loss train": perdida_total / pasos_por_epoca, "log loss val": perdida_val,
                         "accuracy val": accuracy_score(y[idx_val], proba_val >= 0.5), "minutos": (time.time() - inicio) / 60})
        print(f"  fold {k}, época {epoca}: log loss train {historia[-1]['log loss train']:.4f} | val {perdida_val:.4f} | "
              f"accuracy val {historia[-1]['accuracy val']:.4f} | {historia[-1]['minutos']:.1f} min", flush=True)
        if perdida_val < mejor["log_loss"]:
            mejor = {"log_loss": perdida_val, "epoca": epoca, "proba_val": proba_val,
                     "proba_test": predecir(modelo, ids_test, mascara_test)}
    del modelo, optimizador
    torch.cuda.empty_cache()
    return idx_val, mejor, historia

In [ ]:
resumen_folds = {}
for k in range(1, 6):
    archivo_oof = os.path.join(RUTA_SALIDA, f"fold_{k}_oof.csv")
    archivo_test = os.path.join(RUTA_SALIDA, f"fold_{k}_test.csv")
    archivo_info = os.path.join(RUTA_SALIDA, f"fold_{k}_info.json")
    if all(os.path.exists(a) for a in (archivo_oof, archivo_test, archivo_info)):
        print(f"Fold {k}: ya estaba guardado, se lee de {RUTA_SALIDA}")
        with open(archivo_info) as f:
            resumen_folds[k] = json.load(f)
        continue
    print(f"=== Fold {k}: entrena con {np.sum(FOLD != k):,} filas, valida con {np.sum(FOLD == k):,}", flush=True)
    inicio = time.time()
    idx_val, mejor, historia = entrenar_fold(k)
    proba_val = mejor["proba_val"]
    pd.DataFrame({"fila": idx_val, "fold": k, "proba_used": proba_val}).to_csv(archivo_oof, index=False)
    pd.DataFrame({"fila": np.arange(len(test)), "proba_used": mejor["proba_test"]}).to_csv(archivo_test, index=False)
    resumen_folds[k] = {"accuracy": accuracy_score(y[idx_val], proba_val >= 0.5), "auc": roc_auc_score(y[idx_val], proba_val),
                        "log_loss": mejor["log_loss"], "mejor_epoca": mejor["epoca"], "minutos": (time.time() - inicio) / 60,
                        "historia": historia}
    with open(archivo_info, "w") as f:
        json.dump(resumen_folds[k], f, indent=1)
    r = resumen_folds[k]
    print(f"Fold {k}: accuracy {r['accuracy']:.4f} | AUC {r['auc']:.4f} | log loss {r['log_loss']:.4f} | "
          f"mejor época {r['mejor_epoca']} | {r['minutos']:.1f} min | guardado en {archivo_oof}", flush=True)

tabla = pd.DataFrame({k: {c: v for c, v in r.items() if c != "historia"} for k, r in resumen_folds.items()}).T
tabla.index.name = "fold"
display(tabla.round(4))
print(f"Tiempo total de entrenamiento: {tabla['minutos'].sum():.1f} min")

## Resultados y archivos finales

- **Out-of-fold:** cada fila de train con la predicción del modelo que no la vio.
- **Test:** promedio de los 5 modelos de fold.

Se comparan con el submodelo TF-IDF del proyecto en dos segmentos de vendedores nuevos con una sola publicación:
- **train + test:** el vendedor tiene una sola publicación sumando train y test (por eso tampoco aparece en los otros folds);
- **en su fold** (la definición del proyecto): el vendedor no aparece en los otros folds y tiene una sola publicación en su fold.

In [ ]:
oof = pd.concat([pd.read_csv(os.path.join(RUTA_SALIDA, f"fold_{k}_oof.csv")) for k in range(1, 6)]).sort_values("fila")
assert (oof["fila"].to_numpy() == np.arange(len(train))).all(), "Faltan filas en las predicciones out-of-fold"
proba_test = np.mean([pd.read_csv(os.path.join(RUTA_SALIDA, f"fold_{k}_test.csv"))["proba_used"].to_numpy()
                      for k in range(1, 6)], axis=0)
oof[["fila", "fold", "proba_used"]].to_csv(os.path.join(RUTA_SALIDA, "finetune_titulo_oof.csv"), index=False)
pd.DataFrame({"fila": np.arange(len(test)), "proba_used": proba_test}).to_csv(
    os.path.join(RUTA_SALIDA, "finetune_titulo_test.csv"), index=False)
print(f"Guardados finetune_titulo_oof.csv ({len(oof):,} filas) y finetune_titulo_test.csv ({len(test):,} filas) en {RUTA_SALIDA}")

proba = oof["proba_used"].to_numpy()
vendedor_train = train["seller_id"].astype(str)
vendedor_test = test["seller_id"].astype(str)
publicaciones_total = pd.concat([vendedor_train, vendedor_test]).value_counts()
en_su_fold = pd.Series(1, index=vendedor_train.index).groupby([vendedor_train, FOLD]).transform("size")
en_otros_folds = vendedor_train.map(vendedor_train.value_counts()) - en_su_fold
SEGMENTOS = {
    "total": np.ones(len(y), dtype=bool),
    "nuevo con una publicación (train + test)": (vendedor_train.map(publicaciones_total) == 1).to_numpy(),
    "nuevo con una publicación en su fold (definición del proyecto)": ((en_otros_folds == 0) & (en_su_fold == 1)).to_numpy(),
}
filas = {}
for nombre, m in SEGMENTOS.items():
    filas[nombre] = {"filas": int(m.sum()), "accuracy fine-tuning": accuracy_score(y[m], proba[m] >= 0.5),
                     "accuracy TF-IDF": REFERENCIA_TFIDF[nombre]["accuracy"],
                     "AUC fine-tuning": roc_auc_score(y[m], proba[m]), "AUC TF-IDF": REFERENCIA_TFIDF[nombre]["auc"]}
resultados = pd.DataFrame(filas).T
resultados["diferencia accuracy (puntos)"] = (resultados["accuracy fine-tuning"] - resultados["accuracy TF-IDF"]) * 100
resultados["diferencia AUC"] = resultados["AUC fine-tuning"] - resultados["AUC TF-IDF"]
display(resultados.round(4))
print(f"Log loss out-of-fold: {log_loss(y, np.clip(proba, 1e-7, 1 - 1e-7)):.4f} | modelo {MODELO} | texto {TEXTO}")
if MAX_FILAS:
    print(f"(prueba con {MAX_FILAS} filas: los números no son comparables con el TF-IDF)")